# 04 — A tool delivered through MCP
MCP changes how the tool is connected. CrewAI still gives the agent a tool.
The local server exposes only a read-only annual ROI calculation.



## Problem

A local tool and a separately hosted capability need different connection lifecycles.


## Concept

MCP supplies a standard tool connection; CrewAI still presents a tool to the agent.

MCP is a transport and discovery boundary, not a new reasoning model. A separate server publishes a tool; CrewAI's adapter converts the discovered capability into a tool an Agent can use.


## Abstraction

CrewAI adapter ↔ stdio MCP server → annual_roi function. This is the smallest new layer introduced by this lesson; the earlier layers remain in place.


## Architecture

```text
Crew Task ──▶ Agent ──▶ MCPServerAdapter
                              │ starts stdio subprocess
                              ▼
                        FastMCP server ──▶ annual_roi
                              │ JSON tool result
                              └───────────────▶ Agent answer
```

Follow the arrows while executing the cells. A tool result or validation failure is an observation, not an instruction that the model may bypass.


## Step 1 — Configure a local stdio server process

Configure a local stdio server process; this file is the client side.


In [ ]:
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()
sys.path.insert(0, str(ROOT))

import workshop  # configure project-local CrewAI storage
from crewai import Agent, Crew, Process, Task
from crewai_tools import MCPServerAdapter
from mcp import StdioServerParameters

from workshop.common import model

server = StdioServerParameters(
    command=sys.executable,
    args=[str(ROOT / "workshop" / "mcp_server.py")],
)



## Connect, inspect, and use the remote tool
The context manager starts and stops the stdio server. Never put a normal
print statement on the server's stdout; that channel carries MCP messages.



## Step 2 — Discover the tool inside the adapter's connection lifetime, then attach it to the CrewAI agent

Discover the tool inside the adapter's connection lifetime, then attach it to the CrewAI agent.


In [ ]:
with MCPServerAdapter(server) as tools:
    print("MCP tools:", [item.name for item in tools])
    analyst = Agent(
        role="ROI Analyst",
        goal="Calculate a planning scenario through MCP",
        backstory="You distinguish estimates from measured outcomes.",
        llm=model(),
        tools=tools,
        max_iter=3,
        allow_delegation=False,
    )
    task = Task(
        description="Use the MCP annual_roi tool with benefit 45000 and cost 30000.",
        expected_output="Scenario ROI percentage with a caveat.",
        agent=analyst,
    )
    print(Crew(agents=[analyst], tasks=[task], process=Process.sequential).kickoff().raw)


## Execution flow

CrewAI adapter ↔ stdio MCP server → annual_roi function. Read the printed result together with the task, selected tool, state, or trace. If the result differs, locate the first boundary where the observed value changed.


## Failure mode to know about

The adapter's context manager owns the server connection. A stray server `print()` corrupts stdio protocol messages; a missing tool name means discovery failed before the LLM ran.


## Production implication

Treat an MCP server as a separate trust boundary: pin the server, inspect discovered tools, constrain filesystem/network access, and close connections reliably.


## Your turn

List the server's tools, then compare their names with the local tool lesson.

**Checkpoint:** The adapter discovers annual_roi and closes its subprocess after use.
